# 01 — Source schema and licence audit

**Objective.** Audit source schemas, row counts, missingness, invalid dates, and the archived source licence before event construction.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

Permitted data blocks: raw source only. The notebook archives the licence but does not make a new legal interpretation.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("01", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import json, shutil
import pandas as pd
from cruxvc.io import read_json, write_json, write_table
from cruxvc.schema import normalize_source_tables, parse_date_series, read_source_csv, schema_audit

source_manifest = read_json(P.protocol / "source_manifest.json")
raw_dir = P.raw / "crunchbase_october_2013"
file_map = {
    "companies": raw_dir / "crunchbase-companies.csv",
    "rounds": raw_dir / "crunchbase-rounds.csv",
    "investments": raw_dir / "crunchbase-investments.csv",
    "acquisitions": raw_dir / "crunchbase-acquisitions.csv",
}
CTX.recorder.inputs.extend([P.protocol / "source_manifest.json", *file_map.values()])
raw = {name: read_source_csv(path) for name, path in file_map.items()}

In [ ]:
companies, rounds, investments, acquisitions = normalize_source_tables(
    raw["companies"], raw["rounds"], raw["investments"], raw["acquisitions"]
)
normalized = {"companies": companies, "rounds": rounds, "investments": investments, "acquisitions": acquisitions}
audit = {name: schema_audit(frame, name) for name, frame in normalized.items()}
date_specs = {"rounds": "funded_at", "investments": "funded_at", "acquisitions": "acquired_at"}
date_rows = []
for name, column in date_specs.items():
    original = normalized[name][column]
    parsed = parse_date_series(original)
    date_rows.append({
        "table": name,
        "date_column": column,
        "nonempty_source_dates": int(original.notna().sum()),
        "unparseable_nonempty_dates": int((original.notna() & parsed.isna()).sum()),
        "minimum_parsed_date": parsed.min(),
        "maximum_parsed_date": parsed.max(),
        "after_administrative_cutoff": int(parsed.gt(pd.Timestamp(CFG["source"]["administrative_cutoff"])).sum()),
    })
date_audit = pd.DataFrame(date_rows)

In [ ]:
expected_rows = CFG["source"].get("expected_raw_rows", {})
if source_manifest["all_expected_hashes_match"]:
    for filename, expected in expected_rows.items():
        table = next(name for name, path in file_map.items() if path.name == filename)
        observed = len(raw[table])
        if observed != int(expected):
            raise RuntimeError(f"Row-count mismatch for {filename}: expected {expected}, observed {observed}")

schema_path = write_json(audit, P.audits / "01_source_schema_audit.json")
dates_path = write_table(date_audit, P.audits / "01_date_quality_audit.csv")
missing_rows = []
for table, frame in normalized.items():
    for column in frame.columns:
        missing_rows.append({"table": table, "column": column, "missing_n": int(frame[column].isna().sum()), "missing_rate": float(frame[column].isna().mean())})
missing_path = write_table(pd.DataFrame(missing_rows), P.audits / "01_missingness.csv")

In [ ]:
checkout = Path(source_manifest["source_checkout"]["path"])
licence_candidates = list(checkout.glob("LICENSE*"))
if not licence_candidates:
    raise FileNotFoundError("Source licence file was not found in the checked-out mirror")
licence_copy = P.audits / "Crunchbase_2013_source_LICENSE.txt"
shutil.copy2(licence_candidates[0], licence_copy)
CTX.recorder.complete([schema_path, dates_path, missing_path, licence_copy])
print(date_audit.to_string(index=False))